# Quiescence SEE: 500 ms, tek toplu 100 oyun

Aynı sabit commit'in iki ayrı checkout'u karşılaştırılır: yeni tarafta
**QuiescenceSEE=true**, referans tarafta **QuiescenceSEE=false**.
Önce Kaggle'da tüm doğruluk testleri ve SEE kapalı / 4068a04 eşdeğerlik
kontrolü geçer. Ardından tek çağrıda 100 maç oynanır. Hiçbir hamle elenmez;
süre politikası ve beraberlik politikası bu deneyde değiştirilmez.

`codex/quiescence-see` dalından bu notebook'u içe alın. Kaggle Internet açık
olsun; Secrets altında Contents yazma yetkili `GITHUB_TOKEN` ekleyin.
Yeni notebook'a geçerken eski Python importlarını temizlemek için oturumu
başından başlatın. Tüm testler ve maçlar yalnız Kaggle'da çalışır.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/quiescence-see"
WORKSPACE = "/kaggle/working/acemi"
SESSION_ID = "quiescence-see-500ms-100games-v1"
GAMES = 100
CHUNK_GAMES = 100
MOVETIME_MS = 500
HASH_MB = 16
CONCURRENCY = 2
MAX_PLIES = 400
SEED = 1
RETRY_INCOMPLETE = False
REFERENCE_COMMIT = "4068a0459f8119274014cd466270a2ffd3cd30c3"
NEW_OPTIONS = {"QuiescenceSEE": True}
OLD_OPTIONS = {"QuiescenceSEE": False}


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This notebook runs matches only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
if not (workspace / "tools" / "ab_compare.py").exists():
    raise RuntimeError("Import the latest notebook from codex/quiescence-see and rerun setup.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)

def pinned_checkout(path, commit):
    path = Path(path)
    if not path.exists():
        checked(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL, str(path)])
    checked(["git", "-C", str(path), "fetch", "--depth", "1", "origin", commit])
    checked(["git", "-C", str(path), "checkout", "--detach", commit])
    if checked(["git", "-C", str(path), "rev-parse", "HEAD"]) != commit:
        raise RuntimeError("Pinned source mismatch")
    return path

# The first run pins the full branch commit. Resume uses the saved source SHA,
# even after data-only commits have been published to the same branch.
session = workspace / "data" / "ab" / SESSION_ID
if (session / "protocol.json").exists():
    saved = json.loads((session / "protocol.json").read_text())
    CODE_COMMIT = saved["sources"]["new"]["commit"]
    if saved["sources"]["old"]["commit"] != CODE_COMMIT:
        raise RuntimeError("SEE on/off must use the same source commit.")
else:
    CODE_COMMIT = checked(["git", "-C", str(workspace), "rev-parse", "HEAD"])
NEW_COMMIT = OLD_COMMIT = CODE_COMMIT
new_source = pinned_checkout("/kaggle/working/acemi-qsee-on", CODE_COMMIT)
old_source = pinned_checkout("/kaggle/working/acemi-qsee-off", CODE_COMMIT)
reference_source = pinned_checkout("/kaggle/working/acemi-qsee-reference", REFERENCE_COMMIT)
print("PINNED SOURCE:", CODE_COMMIT)
print("NEW:", NEW_OPTIONS, "REFERENCE:", OLD_OPTIONS)
print("100 games in ONE run; 50 paired openings; 500 ms/move; results:", session)


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    paths += [p for p in ("data/runs", "data/index.csv", "data/positions") if (workspace / p).exists()]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Prepare the isolated SEE on/off protocol; completed results are not replayed.
import inspect
from tools import ab_compare as ab
if Path(ab.__file__).resolve() != workspace / "tools" / "ab_compare.py":
    raise RuntimeError("Wrong tools package loaded; restart the Kaggle kernel and Run All.")

if "new_commit" not in inspect.signature(ab.prepare).parameters:
    raise RuntimeError("Stale tools import: restart the Kaggle session and Run All.")

if (session / "protocol.json").exists():
    plan = ab.load_plan(session)
    expected = {"games": GAMES, "movetime_ms": MOVETIME_MS, "hash_mb": HASH_MB,
                "max_plies": MAX_PLIES, "selection_seed": SEED, "chunk_games": CHUNK_GAMES,
                "concurrency": min(CONCURRENCY, os.cpu_count() or 1),
                "feature_options": {"new": NEW_OPTIONS, "old": OLD_OPTIONS}}
    if any(plan[key] != value for key, value in expected.items()):
        raise RuntimeError("Settings differ from saved experiment; use a new SESSION_ID.")
else:
    plan = ab.prepare(session, new_source, old_source, games=GAMES, chunk_games=CHUNK_GAMES,
                      movetime_ms=MOVETIME_MS, concurrency=CONCURRENCY, hash_mb=HASH_MB,
                      max_plies=MAX_PLIES, seed=SEED,
                      new_commit=CODE_COMMIT, old_commit=CODE_COMMIT,
                      new_options=NEW_OPTIONS, old_options=OLD_OPTIONS)
print((session / "SUMMARY.md").read_text())
publish_results(f"data: prepare quiescence SEE 100-game A/B ({SESSION_ID})")


In [ ]:
# Full suites and searches happen HERE in Kaggle, before the match.
command = [sys.executable, "-u", "-m", "tools.qsee_validation", "--session", str(session),
           "--reference", str(reference_source)]
result = subprocess.run(command, cwd=workspace)
try:
    publish_results(f"data: quiescence SEE correctness gate ({SESSION_ID})")
except Exception:
    print("Push failed. Logs remain in Kaggle; the final cell can retry publication without games.")
    raise
if result.returncode:
    raise RuntimeError("Correctness gate failed. No games started; inspect validation/tests.log and status.json.")
print("Correctness gate passed. SEE-off matches 4068a04; next cell runs the 100 games.")


In [ ]:
# The gate must have passed before matches can start.
status = json.loads((session / "validation" / "status.json").read_text())
if not status.get("passed") or status["identity"]["protocol_sha256"] != plan["protocol_sha256"]:
    raise RuntimeError("Kaggle correctness validation has not passed for this protocol.")
# One invocation schedules ALL 100 games. No 10-game batches.
if len(plan["chunks"]) != 1 or plan["chunks"][0]["games"] != 100 or plan["movetime_ms"] != 500:
    raise RuntimeError("This notebook requires one bulk 100-game experiment at 500 ms.")
command = [sys.executable, "-u", "-m", "tools.ab_compare", "run", "--session", str(session), "--chunk", "1"]
if RETRY_INCOMPLETE:
    command.append("--retry-incomplete")
print("Starting ONE 100-game SEE ON/OFF run: 500 ms/move for both sides.", flush=True)
result = subprocess.run(command, cwd=workspace)
try:
    publish_results(f"data: quiescence SEE 100-game results ({SESSION_ID})")
except Exception:
    print("Push failed. Records remain in Kaggle; retry the final cell. It does not start games.")
    raise
if result.returncode:
    raise RuntimeError(f"A/B run stopped, exit={result.returncode}. Inspect the saved log before retrying.")
print((session / "SUMMARY.md").read_text())


In [ ]:
# Safe to rerun after a push failure: it does not start any games.
publish_results(f"data: finalize quiescence SEE 100-game A/B records ({SESSION_ID})")
if (session / "results.json").exists():
    print((session / "SUMMARY.md").read_text())
    result = json.loads((session / "results.json").read_text())
    if result["recorded_attempts"] == GAMES:
        print("quiescence SEE 100-game A/B finished. W/D/L is from the SEE-ON side perspective.")
